# NAFNet — Speckle Noise Reduction & ×2 Super-Resolution

**Pipeline:** `NoisyLR (128×128)` → LogVST → NAFNet (BF16/FP16) → SoftClip → `Prediction (256×256)`

**Loss:** Charbonnier + MS-SSIM + LPIPS + Sobel + Homoscedastic MOO weighting

---
**Before running:** `Runtime → Change runtime type → T4 GPU`

## Step 1 — GPU Check

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1e9
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"GPU name        : {gpu.name}")
    print(f"VRAM            : {vram_gb:.1f} GB")
    print(f"BF16 support    : {bf16_ok}")
    print(f"CUDA version    : {torch.version.cuda}")

    name = gpu.name.lower()
    if 'h100' in name:
        print('\n✅ H100 detected — BF16 + torch.compile + CUDA Graphs fully supported')
    elif 'a100' in name:
        print('\n✅ A100 detected — BF16 + torch.compile supported')
    elif 't4' in name:
        print('\n⚠️  T4 detected — will use FP16, torch.compile disabled (auto-patched below)')
    else:
        print(f'\nℹ️  GPU: {gpu.name}')
else:
    print('\n❌ No GPU found — switch runtime to T4 GPU')

## Step 2 — Mount Drive & Set Up Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, os, shutil

# ── Edit this path to match where you uploaded the zip ──────────────────────
PROJECT_ZIP = "/content/drive/MyDrive/Image-noise-reduction.zip"
PROJECT_DIR = "/content/Image-noise-reduction"
# ────────────────────────────────────────────────────────────────────────────

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

with zipfile.ZipFile(PROJECT_ZIP, 'r') as z:
    z.extractall("/content/")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print("Contents:", os.listdir('.'))

## Step 3 — Install Dependencies

In [ ]:
# torch & torchvision are pre-installed in Colab
!pip install -q lpips pytorch-msssim tqdm pyyaml
print('✅ Dependencies installed')

## Step 4 — Auto-Patch Config for Your GPU

In [ ]:
import sys, yaml, torch
sys.path.insert(0, '.')

CONFIG_PATH = "configs/train.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_properties(0).name.lower()
    bf16_ok  = torch.cuda.is_bf16_supported()
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

    if 'h100' in gpu_name or 'a100' in gpu_name:
        cfg['train']['amp_dtype']   = 'bfloat16'
        cfg['train']['compile']     = True
        cfg['train']['batch_size']  = 16
        print('✅ H100/A100 config: BF16, compile=True, batch_size=16')

    elif 't4' in gpu_name or not bf16_ok:
        cfg['train']['amp_dtype']   = 'float16'
        cfg['train']['compile']     = False   # torch.compile is slow on T4
        cfg['train']['batch_size']  = 8       # T4 has 15 GB VRAM
        print('⚠️  T4 config: FP16, compile=False, batch_size=8')

    else:
        cfg['train']['amp_dtype']   = 'float16' if not bf16_ok else 'bfloat16'
        cfg['train']['compile']     = False
        cfg['train']['batch_size']  = 8 if vram_gb < 20 else 16
        print(f'ℹ️  Generic GPU config: {cfg["train"]["amp_dtype"]}, batch_size={cfg["train"]["batch_size"]}')

else:
    cfg['train']['amp_dtype']  = 'float32'
    cfg['train']['compile']    = False
    cfg['train']['batch_size'] = 4
    print('❌ CPU mode (slow): float32, batch_size=4')

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"\nFinal config → amp_dtype={cfg['train']['amp_dtype']}, "
      f"batch_size={cfg['train']['batch_size']}, compile={cfg['train']['compile']}")

## Step 5 — Sanity Check (Forward Pass)

In [ ]:
import torch, sys
sys.path.insert(0, '.')

from models.nafnet    import NAFNet
from data.transforms  import LogVST
from data.dataset     import SpeckleDataset

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log_vst = LogVST(eps=1e-3).to(device)

# Build full-size model (matches train.yaml)
model = NAFNet(
    in_ch=1, width=64,
    enc_blocks=[2, 2, 4, 8],
    dec_blocks=[2, 2, 2, 2],
    upscale=2,
).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters : {n_params / 1e6:.2f} M')

# Test with a real sample from the dataset
ds  = SpeckleDataset('dataset/train/train/GT', 'dataset/train/train/NoisyLR', indices=[0])
lr, gt = ds[0]
lr = lr.unsqueeze(0).to(device)   # (1, 1, 128, 128)
gt = gt.unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(log_vst(lr))

print(f'LR  input range  : [{lr.min():.4f}, {lr.max():.4f}]  shape={tuple(lr.shape)}')
print(f'GT  range        : [{gt.min():.4f}, {gt.max():.4f}]  shape={tuple(gt.shape)}')
print(f'Pred range       : [{pred.min():.4f}, {pred.max():.4f}]  shape={tuple(pred.shape)}')
assert pred.shape == (1, 1, 256, 256), 'Shape mismatch!'
assert not pred.isnan().any(), 'NaN in output!'
print('\n✅ Sanity check PASSED')

## Step 6 — Loss Function Check

In [ ]:
from losses.compound_loss import CompoundLoss

criterion = CompoundLoss(
    lambda_charb=1.0, lambda_msssim=0.1,
    lambda_lpips=0.05, lambda_sobel=0.01
).to(device)

# Use the prediction and GT from the sanity check above
loss, loss_dict = criterion(pred, gt)

print('Loss components:')
for k, v in loss_dict.items():
    print(f'  {k:12s}: {v:.6f}')

assert not torch.isnan(loss), 'NaN in total loss!'
print('\n✅ Loss check PASSED')

## Step 7 — Training

### Where are checkpoints saved?

| Location | Survives session end? | Notes |
|---|---|---|
| `/content/.../checkpoints/best_psnr.pth` | ❌ **Deleted** when Colab resets | Local fast write |
| `/content/drive/MyDrive/.../checkpoints/` | ✅ **Permanent** on your Drive | Synced after every epoch |

The training loop copies `best_psnr.pth` and `best_lpips.pth` to Drive after **every epoch**.
If your session crashes or times out, run the **Restore checkpoint** cell below, then re-run the training cell — it will pick up from the best epoch automatically.


In [ ]:
# ── Restore checkpoint from Drive after a crash / new Colab session ──────────
# Run this cell BEFORE the training cell when resuming a previous run.
# Skip it on a fresh start.
import os, shutil, torch

DRIVE_CKPT = '/content/drive/MyDrive/Image-noise-reduction/checkpoints/'
LOCAL_CKPT = 'checkpoints/'
os.makedirs(LOCAL_CKPT, exist_ok=True)

restored = []
for fname in ['best_psnr.pth', 'best_lpips.pth']:
    src = os.path.join(DRIVE_CKPT, fname)
    dst = os.path.join(LOCAL_CKPT, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        restored.append(fname)

if restored:
    for fname in restored:
        ckpt = torch.load(os.path.join(LOCAL_CKPT, fname), map_location='cpu')
        print(f'Restored {fname}:  epoch={ckpt["epoch"]:03d} | '
              f'PSNR={ckpt["psnr"]:.3f} dB | SSIM={ckpt["ssim"]:.4f} | LPIPS={ckpt["lpips"]:.4f}')
    print('\nThe training cell will resume automatically from the best saved epoch.')
else:
    print('No checkpoints found in Drive — starting fresh training from epoch 1.')


In [ ]:
# ── Quick overfit test (1 batch) — runs in < 2 minutes ──────────────────────
# Verifies the full train loop works before committing to 200 epochs
!python train.py --config configs/train.yaml --overfit-batch

In [ ]:
# ── Full inline training (runs in this cell, progress visible live) ──────────
import sys, os, time, yaml, torch
import torch.nn as nn
from tqdm.notebook import tqdm
sys.path.insert(0, '.')

from data.dataset      import build_dataloaders
from data.transforms   import LogVST
from losses.compound_loss import CompoundLoss
from models.nafnet     import build_model
from utils.metrics     import MetricsAggregator

# ── Config ────────────────────────────────────────────────────────────────────
with open('configs/train.yaml') as f:
    cfg = yaml.safe_load(f)

EPOCHS      = cfg['train']['epochs']
BATCH_SIZE  = cfg['train']['batch_size']
LR          = cfg['train']['lr']
GRAD_CLIP   = cfg['train']['grad_clip']
WARMUP      = cfg['train']['warmup_steps']
AMP_DTYPE   = {'bfloat16': torch.bfloat16,
               'float16':  torch.float16,
               'float32':  torch.float32}[cfg['train']['amp_dtype']]
CKPT_DIR    = cfg['logging']['checkpoint_dir']
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | AMP: {cfg["train"]["amp_dtype"]} | Batch: {BATCH_SIZE}')

# ── Data ──────────────────────────────────────────────────────────────────────
train_loader, val_loader = build_dataloaders(
    gt_dir          = cfg['data']['train_gt_dir'],
    lr_dir          = cfg['data']['train_lr_dir'],
    batch_size      = BATCH_SIZE,
    val_split       = cfg['data']['val_split'],
    num_workers     = cfg['data']['num_workers'],
    pin_memory      = cfg['data']['pin_memory'],
    prefetch_factor = cfg['data']['prefetch_factor'],
)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

# ── Model ─────────────────────────────────────────────────────────────────────
model     = build_model(cfg, device)
log_vst   = LogVST(eps=1e-3).to(device)
criterion = CompoundLoss(
    lambda_charb   = cfg['loss']['lambda_charb'],
    lambda_msssim  = cfg['loss']['lambda_msssim'],
    lambda_lpips   = cfg['loss']['lambda_lpips'],
    lambda_sobel   = cfg['loss']['lambda_sobel'],
    charb_eps      = cfg['loss']['charb_eps'],
).to(device)

# ── Optimiser: two groups — model params + MOO log_vars ───────────────────────
optimizer = torch.optim.AdamW([
    {'params': model.parameters(),     'weight_decay': cfg['train']['weight_decay']},
    {'params': criterion.log_vars,     'weight_decay': 0.0, 'lr': LR * 0.1},
], lr=LR)

# ── Cosine LR schedule with linear warmup ─────────────────────────────────────
import math
total_steps = EPOCHS * len(train_loader)

def lr_lambda(step):
    if step < WARMUP:
        return step / max(1, WARMUP)
    progress = (step - WARMUP) / max(1, total_steps - WARMUP)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Tracking ──────────────────────────────────────────────────────────────────
best_psnr  = -float('inf')
best_lpips =  float('inf')
history    = {'epoch': [], 'loss': [], 'psnr': [], 'ssim': [], 'lpips': []}
global_step = 0
scaler = torch.cuda.amp.GradScaler(enabled=(AMP_DTYPE == torch.float16))

# ── Resume from checkpoint if one exists ──────────────────────────────────────
START_EPOCH = 1
best_psnr   = -float('inf')
best_lpips  =  float('inf')

resume_path = os.path.join(CKPT_DIR, 'best_psnr.pth')
if os.path.exists(resume_path):
    ckpt_resume = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt_resume['model'])
    START_EPOCH = ckpt_resume['epoch'] + 1
    best_psnr   = ckpt_resume['psnr']
    best_lpips  = ckpt_resume.get('lpips', float('inf'))
    # Advance scheduler to correct position
    for _ in range((START_EPOCH - 1) * len(train_loader)):
        scheduler.step()
    print(f'Resumed from epoch {ckpt_resume["epoch"]:03d} | '
          f'Best PSNR so far: {best_psnr:.3f} dB')
else:
    print('No checkpoint found — training from scratch.')


# ─────────────────────────────────────────────────────────────────────────────
# Training loop
# ─────────────────────────────────────────────────────────────────────────────
for epoch in range(START_EPOCH, EPOCHS + 1):
    model.train()
    criterion.train()
    epoch_loss = 0.0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f'Epoch {epoch:03d}/{EPOCHS}', leave=False)
    for lr_imgs, gt_imgs in pbar:
        lr_imgs = lr_imgs.to(device, non_blocking=True)
        gt_imgs = gt_imgs.to(device, non_blocking=True)

        # Log-VST variance stabilisation
        lr_imgs = log_vst(lr_imgs)

        # BF16/FP16 forward pass
        use_amp = AMP_DTYPE != torch.float32
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=use_amp):
            pred = model(lr_imgs)
            loss, loss_dict = criterion(pred, gt_imgs)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + [criterion.log_vars],
            max_norm=GRAD_CLIP,
        )
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        global_step += 1
        epoch_loss  += loss_dict['total']

        pbar.set_postfix(
            loss=f"{loss_dict['total']:.4f}",
            charb=f"{loss_dict['charb']:.4f}",
            lpips=f"{loss_dict['lpips']:.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}",
        )

    avg_loss = epoch_loss / len(train_loader)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    agg = MetricsAggregator(lpips_device=str(device))
    with torch.no_grad():
        for lr_imgs, gt_imgs in val_loader:
            lr_imgs = lr_imgs.to(device, non_blocking=True)
            gt_imgs = gt_imgs.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=use_amp):
                pred = model(log_vst(lr_imgs))
            agg.update(pred.cpu().float(), gt_imgs.cpu().float())

    metrics = agg.compute()
    psnr, ssim, lpips_val = metrics['psnr'], metrics['ssim'], metrics['lpips']

    print(
        f"Ep {epoch:03d} | loss={avg_loss:.4f} | "
        f"PSNR={psnr:.3f} dB | SSIM={ssim:.4f} | LPIPS={lpips_val:.4f} | "
        f"{time.time()-t0:.0f}s | "
        f"σ=[{criterion.log_vars.exp().sqrt().detach().cpu().numpy().round(3)}]"
    )

    # ── Checkpointing ─────────────────────────────────────────────────────────
    state = {'epoch': epoch, 'model': model.state_dict(),
             'psnr': psnr, 'ssim': ssim, 'lpips': lpips_val}

    if psnr > best_psnr:
        best_psnr = psnr
        torch.save(state, os.path.join(CKPT_DIR, 'best_psnr.pth'))
        print(f'  ✓ best_psnr.pth saved  (PSNR={best_psnr:.3f} dB)')

    if lpips_val < best_lpips:
        best_lpips = lpips_val
        torch.save(state, os.path.join(CKPT_DIR, 'best_lpips.pth'))
        print(f'  ✓ best_lpips.pth saved (LPIPS={best_lpips:.4f})')

    # ── Copy checkpoint to Drive (keeps progress safe across sessions) ─────────
    drive_ckpt = '/content/drive/MyDrive/Image-noise-reduction/checkpoints/'
    os.makedirs(drive_ckpt, exist_ok=True)
    import shutil
    for fname in ['best_psnr.pth', 'best_lpips.pth']:
        src = os.path.join(CKPT_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, drive_ckpt)

    history['epoch'].append(epoch)
    history['loss'].append(avg_loss)
    history['psnr'].append(psnr)
    history['ssim'].append(ssim)
    history['lpips'].append(lpips_val)

print(f'\n✅ Training complete.  Best PSNR: {best_psnr:.3f} dB | Best LPIPS: {best_lpips:.4f}')

## Step 8 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titles  = ['Train Loss', 'PSNR (dB)', 'SSIM', 'LPIPS']
keys    = ['loss', 'psnr', 'ssim', 'lpips']
colors  = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']
arrows  = ['↓', '↑', '↑', '↓']

for ax, title, key, color, arrow in zip(axes, titles, keys, colors, arrows):
    ax.plot(history['epoch'], history[key], color=color, linewidth=2)
    ax.set_title(f'{title} {arrow}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('NAFNet Training Progress', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

## Step 9 — Visual Inspection

In [ ]:
import torch, sys
import matplotlib.pyplot as plt
import torch.nn.functional as F
sys.path.insert(0, '.')

from data.dataset     import SpeckleDataset
from data.transforms  import LogVST
from models.nafnet    import NAFNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load best PSNR model ──────────────────────────────────────────────────────
model_vis = NAFNet(in_ch=1, width=64, enc_blocks=[2,2,4,8],
                   dec_blocks=[2,2,2,2], upscale=2).to(device)
ckpt = torch.load('checkpoints/best_psnr.pth', map_location=device)
model_vis.load_state_dict(ckpt['model'])
model_vis.eval()
print(f"Loaded checkpoint — PSNR={ckpt['psnr']:.3f} dB, "
      f"SSIM={ckpt['ssim']:.4f}, LPIPS={ckpt['lpips']:.4f}")

log_vst = LogVST(eps=1e-3).to(device)

# ── Sample 4 images from val set ─────────────────────────────────────────────
ds = SpeckleDataset(
    'dataset/train/train/GT', 'dataset/train/train/NoisyLR',
    indices=[100, 200, 300, 400]   # pick some indices
)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
col_titles = ['NoisyLR (bicubic ×2)', 'NAFNet Prediction', 'Ground Truth']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontweight='bold', fontsize=11)

for row in range(4):
    lr, gt = ds[row]
    lr_dev = lr.unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model_vis(log_vst(lr_dev)).squeeze(0).cpu()

    # Bicubic upsample LR for comparison
    lr_up = F.interpolate(lr.unsqueeze(0), size=(256, 256),
                          mode='bicubic', align_corners=False).squeeze(0)

    for col, img in enumerate([lr_up, pred, gt]):
        axes[row, col].imshow(img.squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')

plt.suptitle('NAFNet Restoration Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visual_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visual_results.png')

## Step 10 — Run Inference on Test Set

In [ ]:
# Runs the full async inference pipeline (B=64, 16-worker disk write)
!python infer.py --config configs/infer.yaml --benchmark

In [ ]:
import os, numpy as np

pred_dir = 'predictions/'
files    = sorted(os.listdir(pred_dir))
npys     = [f for f in files if f.endswith('.npy')]
print(f'Saved {len(npys)} predictions in {pred_dir}')

# Verify output format
sample = np.load(os.path.join(pred_dir, npys[0]))
print(f'Sample shape : {sample.shape}  dtype: {sample.dtype}')
print(f'Sample range : [{sample.min():.4f}, {sample.max():.4f}]')
assert sample.shape == (256, 256), f'Expected (256,256), got {sample.shape}'
assert sample.dtype == np.float32
print('✅ Predictions verified')

## Step 11 — Save Predictions to Drive

Copy the `predictions/` folder back to Google Drive for download.

In [ ]:
import shutil, os

DRIVE_OUT = '/content/drive/MyDrive/Image-noise-reduction/predictions/'
os.makedirs(DRIVE_OUT, exist_ok=True)

for fname in os.listdir('predictions/'):
    shutil.copy(os.path.join('predictions', fname), DRIVE_OUT)

print(f'✅ Predictions copied to Drive: {DRIVE_OUT}')
print(f'   Files: {len(os.listdir(DRIVE_OUT))}')

## [H100 Only] — Verify H100 Runtime & Enable CUDA Graphs

Run this cell **only** if your submission environment provides an H100.

In [ ]:
import torch, yaml

# ── H100 Verification ────────────────────────────────────────────────────────
assert torch.cuda.is_available(), 'No GPU found'
gpu = torch.cuda.get_device_properties(0)

print(f'GPU name    : {gpu.name}')
print(f'VRAM        : {gpu.total_memory / 1e9:.1f} GB')
print(f'BF16 support: {torch.cuda.is_bf16_supported()}')
print(f'SM count    : {gpu.multi_processor_count}')

is_h100 = 'h100' in gpu.name.lower()
if is_h100:
    print('\n✅ H100 confirmed — enabling CUDA Graphs for final inference')

    # Patch infer.yaml to use CUDA Graphs
    with open('configs/infer.yaml') as f:
        infer_cfg = yaml.safe_load(f)

    infer_cfg['infer']['compile_mode']    = 'max-autotune'
    infer_cfg['infer']['use_cuda_graphs'] = True

    with open('configs/infer.yaml', 'w') as f:
        yaml.dump(infer_cfg, f, default_flow_style=False)

    print('  infer.yaml → compile_mode=max-autotune, use_cuda_graphs=True')

    # Verify BF16 throughput
    dummy = torch.randn(64, 1, 128, 128, device='cuda')
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        import time
        torch.cuda.synchronize()
        t = time.perf_counter()
        for _ in range(10):
            _ = torch.nn.functional.conv2d(
                dummy.bfloat16(),
                torch.randn(64, 1, 3, 3, device='cuda', dtype=torch.bfloat16),
                padding=1
            )
        torch.cuda.synchronize()
    print(f'  BF16 conv warmup: {(time.perf_counter()-t)*1000:.1f} ms (10 iters)')
else:
    print(f'\n⚠️  This is NOT an H100 ({gpu.name}). Keep dev settings.')